# Linear-grid version of Figure 3.6: $r=1$

Repeat the current Figure 3.6 fixed-sample joint $(N,\lambda_N)$ experiment, changing only the bandwidth grid to

$$h_i=0.01+\frac{i-1}{N-1}(2-0.01),\qquad i=1,\ldots,N.$$

The sample is the same 200 standard-normal draws from NumPy's `default_rng(20260823)`. The interval remains $[0.01,2]$, $N=100,200,\ldots,3000$, and $\lambda_N\in\{1,N^{1/2},N,N^{3/2},N^2,N^3\}$. The grid helper calls this `beta=0`, corresponding to thesis $r=1$ through $\beta=2(1-r)$; the divergence remains one half of the squared $L^2$ distance.

Sources: [current Figure 3.6 and grid definition, Eq. `eq:def_r_linear`](../../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex); [original experiment, initial setup and sweep](experiment_1_joint_N_lambda_limit.ipynb); [grid helper](../kde_tools/bandwidths.py).

## Setup

Enable the shared automatic PDF exporter before creating or showing figures. Every displayed figure is saved under `../outputs/pdfs/linear_grid_joint_n_lambda_limit/`. The shared implementation below also saves the sample, full results, reference scan, and validation diagnostics under `../outputs/data/linear_grid_joint_n_lambda_limit/`. Completed grid sizes are reused on reruns.

In [ ]:
%matplotlib inline

from pathlib import Path
import runpy
import sys

KDE_DIR = next(
    candidate.resolve()
    for ancestor in (Path.cwd(), *Path.cwd().parents)
    for candidate in (ancestor, ancestor / "Code" / "Code_kde")
    if (candidate / "kde_tools" / "pdf_export.py").is_file()
)
if str(KDE_DIR) not in sys.path:
    sys.path.insert(0, str(KDE_DIR))

from kde_tools.pdf_export import enable_automatic_pdf_saving
from threadpoolctl import threadpool_limits

NOTEBOOK_NAME = "linear_grid_joint_N_lambda_limit"
pdf_directory = enable_automatic_pdf_saving(NOTEBOOK_NAME)
workflow = runpy.run_path(str(
    KDE_DIR / "outputs/analysis/linear_grid_joint_n_lambda_limit/run_experiment.py"
))


## Run and validate

The empirical reference is the minimizer on the full permitted interval of $\widehat Q_1(h)=\frac12\|\partial_h\widehat f_h\|_2^2$ in the thesis notation (Proposition `prop:main_thm`). Both endpoints and all local minima detected by a 2001-point logarithmic scan are compared. The exact Gaussian formula validates the Fourier divergence on a 25-point bandwidth grid; doubling the frequency intervals at $N=3000$ must preserve all six selections. Eigenpair residuals and gaps are saved for every solve.

Sources: [thesis local coefficient](../../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex); [original experiment's `local_beta2_log_metric`](experiment_1_joint_N_lambda_limit.ipynb), divided by $h^2$ for this grid; [reproducible workflow and validation](../outputs/analysis/linear_grid_joint_n_lambda_limit/run_experiment.py).

In [ ]:
with threadpool_limits(limits=2):
    results, reference, metric_h, metric_q = workflow["run_experiment"]()

print("Continuous reference on [0.01, 2]:", reference)
display(results.pivot(index="N", columns="alpha", values="selected_h"))


## Result from the saved run

| lambda_N | Selected h at N=3000 |
| --- | ---: |
| 1 | 0.98807936 |
| N^(1/2) | 1.06173391 |
| N | 1.70471824 |
| N^(3/2) | 1.91506502 |
| N^2 | 1.97677559 |
| N^3 | 1.99867289 |

The reference minimum on the inspected interval is at the upper endpoint h=2, with Q_1=0.00773333298388. Both endpoints and the detected interior local minimum were compared. All six N=3000 selections were unchanged after doubling the frequency resolution. Maximum eigenpair residual: 6.925e-15.

![Linear-grid result](../outputs/images/linear_grid_joint_n_lambda_limit/linear_grid_joint_N_lambda_limit.png)

[Full numerical results](../outputs/data/linear_grid_joint_n_lambda_limit/results.csv), [reference calculation](../outputs/data/linear_grid_joint_n_lambda_limit/reference.json), and [validation](../outputs/data/linear_grid_joint_n_lambda_limit/validation.json).

The horizontal reference includes the interval endpoints. A reference at h=2 describes this fixed sample and restricted interval; this experiment alone does not establish an interior convergence theorem. Run the following cell to regenerate the plot and its automatic PDF export.


In [ ]:
image_path = workflow["plot_results"](results, reference)
print("PNG:", image_path)
print("Automatic PDFs:", pdf_directory)
